In [1]:
import pandas as pd
import numpy as np
import os
import math
import warnings
import pickle
import hashlib
from collections import Counter
from itertools import islice

import scipy
from scipy import stats
from scipy.stats import linregress, pearsonr, percentileofscore

import statsmodels.api as sm
import seaborn as sns
import matplotlib.pyplot as plt
import shap

from tqdm.notebook import tqdm
from mlxtend.feature_selection import SequentialFeatureSelector as SFS

from sklearn import metrics
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import GradientBoostingRegressor, HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GridSearchCV, train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils import check_random_state, resample

from skopt import BayesSearchCV
from skopt.space import Real, Integer

warnings.filterwarnings("ignore")

c:\Users\eguen\miniconda3\envs\RedLat\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions

In [2]:
import os
import pickle
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import HistGradientBoostingRegressor

def get_best_features_sfs(data, vars_list, target_col="Age",
             pkl_path="best_features_w-Barthel-Cognition_v01.pkl",
             force=True):
  if (not force) and os.path.exists(pkl_path):
    with open(pkl_path, "rb") as f:
      best_features = pickle.load(f)
    print("Loaded best_features from pickle:", best_features)
  else:

    gbr = HistGradientBoostingRegressor()

    # Aplicar SFS hacia adelante usando R2 como métrica
    sfs = SFS(gbr,
         k_features='best', # o un número como 5 o 10 si lo prefieres fijo
         forward=True,
         floating=False,
         scoring='r2',
         cv=5,
         n_jobs=-1,
         verbose=2)

    # Ejecutar SFS
    sfs = sfs.fit(data[vars_list], data[target_col])

    # Obtener las mejores selected variables
    best_features = list(sfs.k_feature_names_)
    with open(pkl_path, "wb") as f:
      pickle.dump(best_features, f)
    print("Computed and saved best_features:", best_features)

  return best_features



In [3]:
def run_nested_cv_hgbr(data_, best_features,
            y_col="Age", diag_col="anydem"):
  from sklearn.model_selection import KFold, GridSearchCV
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import MinMaxScaler
  from sklearn.ensemble import HistGradientBoostingRegressor
  from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
  from scipy.stats import linregress
  import numpy as np
  import pandas as pd
  #import shap
  from sklearn.inspection import permutation_importance

  # Variables
  y = data_[y_col]
  X_selected = data_[best_features] # selected variables

  # Outer CV
  kf = KFold(n_splits=10, shuffle=True, random_state=42)

  # Hyperparameters for Gradient Boosting
  param_grid = {
    "model__max_iter": [300, 400, 500, 600],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1]
  }

  # Results
  r2_scores = []
  r2_adj_scores = [] # nuevo
  y_true_all = []
  y_pred_all = []
  results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])

  # SHAP accumulator
  shap_values_sum = pd.Series(0, index=X_selected.columns)
  perm_values_sum = pd.Series(0, index=X_selected.columns)

  p = X_selected.shape[1] # number of predictors (constant)

  # Nested CV
  for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
    print(f"Fold {fold + 1} (Nested CV)")

    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # Pipeline
    pipeline = Pipeline([
      ("scaler", MinMaxScaler((0.05, 0.95))),
      ("model", HistGradientBoostingRegressor(random_state=42))
    ])

    # Internal GridSearch
    grid_search = GridSearchCV(
      estimator=pipeline,
      param_grid=param_grid,
      scoring="r2",
      cv=5,
      n_jobs=-1,
      verbose=0
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    # Evaluate
    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

    # Adjusted R2 per fold
    n_test = len(y_test)
    if (n_test - p - 1) > 0:
      r2_adj = 1 - (1 - r2) * (n_test - 1) / (n_test - p - 1)
    else:
      r2_adj = np.nan
    r2_adj_scores.append(r2_adj)

    print(f"R² fold {fold + 1}: {r2:.4f} | R² adj: {r2_adj:.4f} (best hyperparameters: {grid_search.best_params_})")

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

    # GAP
    gap_test = y_pred - y_test
    gap_train_all = best_model.predict(X_train) - y_train

    # Filter only CN subjects in train for regression
    train_ids = X_train.index
    diag_train = data_.loc[train_ids, diag_col]
    cn_mask = diag_train == 0

    slope, intercept, _, _, _ = linregress(y_train[cn_mask], gap_train_all[cn_mask])
    corrected_gap = gap_test - (slope * y_test + intercept)

    perm_importance = permutation_importance(
      best_model, X_test, y_test, n_repeats=30, random_state=42, n_jobs=-1
    )

    # Average decrease in R²
    perm_values_mean = perm_importance.importances_mean
    perm_values_sum += pd.Series(perm_values_mean, index=X_selected.columns)

    # Save results of this fold
    result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
    temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
    temp_df['ID'] = X_test.index
    results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

  # r (Pearson correlation)
  r = np.corrcoef(y_true_all, y_pred_all)[0, 1]

  # RMSE
  rmse = np.sqrt(mean_squared_error(y_true_all, y_pred_all))

  # MAE
  mae = mean_absolute_error(y_true_all, y_pred_all)

  # Cohen's f²: f² = R² / (1 - R²)
  f2 = r2 / (1 - r2) if r2 < 1 else np.inf

  # Global R2 and global adjusted R2
  r2_global = r2_score(y_true_all, y_pred_all)
  n_global = len(y_true_all)
  if (n_global - p - 1) > 0:
    r2_adj_global = 1 - (1 - r2_global) * (n_global - 1) / (n_global - p - 1)
  else:
    r2_adj_global = np.nan

  # Final results
  print("\nR² per fold (Nested CV):", r2_scores)
  print("Adjusted R² per fold (Nested CV):", r2_adj_scores)
  print("Average R²:", np.mean(r2_scores), np.std(r2_scores))
  print("Average adjusted R²:", np.nanmean(r2_adj_scores), np.nanstd(r2_adj_scores))
  print("Global R² (all predictions):", r2_global)
  print("Global adjusted R²:", r2_adj_global)

  print("Cohen's f²:", f2)
  print("r (correlation):", r)
  print("RMSE:", rmse)
  print("MAE:", mae)

  # Average permutation importance
  perm_importance_avg = perm_values_sum / kf.get_n_splits()
  perm_importance_avg = perm_importance_avg.sort_values(ascending=False)
  print("\nAverage importance (Permutation Importance):")
  print(perm_importance_avg)

  return (r2_scores, r2_adj_scores, results_labels_df, None, perm_importance_avg, r2_global, r2_adj_global)



In [4]:
def nested_cv_flag_bad_subjects(data_, best_features, y_col="Age", diag_col="anydem"):
  from sklearn.model_selection import KFold, GridSearchCV
  from sklearn.pipeline import Pipeline
  from sklearn.preprocessing import MinMaxScaler
  from sklearn.ensemble import HistGradientBoostingRegressor
  from sklearn.metrics import r2_score
  from scipy.stats import linregress
  import numpy as np
  import pandas as pd

  # Variables
  y = data_[y_col]
  X_selected = data_[best_features]

  kf = KFold(n_splits=10, shuffle=True, random_state=42)

  param_grid = {
    "model__max_iter": [50, 100, 200],
    "model__max_depth": [3, 5, 7],
    "model__learning_rate": [0.01, 0.05, 0.1]
  }

  r2_scores = []
  y_true_all, y_pred_all = [], []
  results_labels_df = pd.DataFrame(columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected', 'ID'])
  suspected_bad_subjects = []

  # Nested CV
  for fold, (train_idx, test_idx) in enumerate(kf.split(X_selected)):
    print(f"Fold {fold + 1} (Nested CV)")

    X_train, X_test = X_selected.iloc[train_idx], X_selected.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline = Pipeline([
      ("scaler", MinMaxScaler((0.05, 0.95))),
      ("model", HistGradientBoostingRegressor(random_state=42))
    ])

    grid_search = GridSearchCV(
      estimator=pipeline,
      param_grid=param_grid,
      scoring="r2",
      cv=5,
      n_jobs=-1,
      verbose=0
    )

    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_

    y_pred = best_model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    r2_scores.append(r2)

    print(f"R² fold {fold + 1}: {r2:.4f}")

    y_true_all.extend(y_test)
    y_pred_all.extend(y_pred)

    gap_test = y_pred - y_test
    gap_train = best_model.predict(X_train) - y_train
    slope, intercept, _, _, _ = linregress(y_train, gap_train)
    corrected_gap = gap_test - (slope * y_test + intercept)

    result = np.column_stack((y_test, y_pred, gap_test, corrected_gap))
    temp_df = pd.DataFrame(result, columns=['y_labels', 'y_pred', 'GAP', 'GAP_corrected'])
    temp_df['ID'] = X_test.index

    results_labels_df = pd.concat([results_labels_df, temp_df], ignore_index=True)

    # Identify subjects that greatly reduce R² in this fold
    if r2 < 0.25:
      individual_errors = np.abs(y_test - y_pred)
      threshold = np.percentile(individual_errors, 95) # top 10% worst
      bad_subjects_fold = X_test.index[individual_errors >= threshold].tolist()
      suspected_bad_subjects.extend(bad_subjects_fold)

  # Filter only those that are NOT CN
  non_cn_bad_subjects = [
    subj for subj in set(suspected_bad_subjects)
    if data_.loc[subj, diag_col] != 0
  ]

  print(f"Subjects identified (NOT CN) with negative impact: {len(non_cn_bad_subjects)}")

  # Create new dataframe without those NOT CN subjects
  data_filtered = data_.drop(index=non_cn_bad_subjects)
  print(f"New filtered dataframe: {data_filtered.shape}")

  # Final results
  print("\nAverage R²:", np.mean(r2_scores), np.std(r2_scores))
  print("Global R²:", r2_score(y_true_all, y_pred_all))

  return (data_filtered, non_cn_bad_subjects, r2_scores, results_labels_df)


## All vars model

### Load data

In [5]:
data = pd.read_parquet('../../Data/data.parquet')

In [6]:
vars_list = ['Family_dementia_n',
      'Family_dementia_any', 'Sex_1F_2M', 'Education', 'Assets', 
      'Ataxia', 'Bradykinesia', 
      'Hypertension_1Y_0N', 'Heart_Disease_1Y_0N', 'Vision_problems',
      'Audition_problems',
       'pulse_pressure',
       'orthostatic_drop',
      'Stroke_1Y_0N', 'TIA_1Y_0N', 'Back_diseases', 
       'Medic_treated',
      'Arthritis', 'Cough',
      'Breathlessness', 'Breath_problems', 'Angina', 'Stomach_problems',
      'Faints', 'Paralysis', 'Limiting_illnesses', 'Skin_disorder', 'Pain',
      'Alcohol_1Y_0N', 'Physical_activities',
      'Walk_1Y_0N', 'Exercise_increase', 'Medic_visits', 'Medications_1Y_0N']

### SFS all subjects

In [7]:
best_features = get_best_features_sfs(data, vars_list,
                   target_col="Age",
                   pkl_path="../../SFS/best_features-wo-used-diag-vars-and-funct-vars_all-subjects_onlyCN.pkl",
                   force=False)


Loaded best_features from pickle: ['Family_dementia_any', 'Sex_1F_2M', 'Education', 'Ataxia', 'Bradykinesia', 'Hypertension_1Y_0N', 'Heart_Disease_1Y_0N', 'Audition_problems', 'pulse_pressure', 'orthostatic_drop', 'Stroke_1Y_0N', 'TIA_1Y_0N', 'Back_diseases', 'Arthritis', 'Cough', 'Breathlessness', 'Breath_problems', 'Stomach_problems', 'Faints', 'Limiting_illnesses', 'Pain', 'Physical_activities', 'Walk_1Y_0N', 'Exercise_increase', 'Medications_1Y_0N']


In [8]:
## Ensure these variables are included
best_features.append('Sex_1F_2M')
best_features.append('Education')

best_features = list(np.unique(best_features))

### BBAGs model-all subjects

In [9]:
r2_scores, r2_adj_scores, results_df, shap_imp, perm_imp, r2_global, r2_adj_global = run_nested_cv_hgbr(
  data_=data,
  best_features=best_features,
  y_col="Age",
  diag_col="anydem"
)


Fold 1 (Nested CV)
R² fold 1: 0.1670 | R² adj: 0.1484 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
Fold 2 (Nested CV)
R² fold 2: 0.1542 | R² adj: 0.1353 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
Fold 3 (Nested CV)
R² fold 3: 0.1505 | R² adj: 0.1316 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
Fold 4 (Nested CV)
R² fold 4: 0.1727 | R² adj: 0.1542 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
Fold 5 (Nested CV)
R² fold 5: 0.1555 | R² adj: 0.1366 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
Fold 6 (Nested CV)
R² fold 6: 0.2027 | R² adj: 0.1850 (best hyperparameters: {'model__learning_rate': 0.05, 'model__max_depth': 3, 'model__max_iter': 300})
Fold 7 (Nested CV)
R² fold 7: 0.1535 | R² adj: 0.1346 (best hype

In [10]:
results_df['y_pred_corrected'] = results_df['y_labels'] + results_df['GAP_corrected']
results_df = results_df.set_index('ID')

results_df = results_df.sort_index()

df_final = pd.concat([data[['uid', 'date', 'Age', 'anydem', 'dsmcase', 'cogcase', 'mci', 'Countries'] + best_features], results_df], axis = 1)

In [11]:
df_final.to_parquet('../../Results/wo-used-diag-vars-and-funct-vars_all-subjects_onlyCN.parquet')

perm_imp.to_frame(name="perm_importance").to_parquet("../../Results/wo-used-diag-vars-and-funct-vars_all-subjects-perm_imp_onlyCN.parquet")


import pickle

out = {
  "r2_scores": r2_scores,
  "r2_adj_scores": r2_adj_scores,
  "r2_global": r2_global,
  "r2_adj_global": r2_adj_global
}

with open("../../Results/r2_wo-used-diag-vars-and-funct-vars_all-subjects_onlyCN.pkl", "wb") as f:
  pickle.dump(out, f)
